# Infaction v0.4.8 fully wired pipeline — 2026-11-22

# Infaction v0.4.8 — WORLD / OWNERSHIP / HIDDEN / RESTRICTED / GATE
**Run all four cells using Runtime → Restart session and run all.** The final cell downloads one complete JSON with six full Factions, geographic Place hierarchy, preserve-first political ownership, Hidden/Restricted status and separate Gate/Checkpoint access nodes. Full source data uses 118/118 Neutral static v2.1 entries.

**Integrated selection v0.4.8:** Effective 71 Location types (Town Hall in, Fortress/Palace/Mine/Factory out), six approved score modifiers, hard Function minimum repair, and the four political post-choice zero-Proximity exceptions are wired into current Faction → Zone → Location → Placement → City → Site → ownership/access/naming. This is an executable integration implementation; a clean hosted-Colab end-to-end run is still unverified. Faction-Type rank thresholds and Site ownership cutoff remain provisional.
**Rules:** preserve existing Single/Dual on Zone/Location; fill only none from new all-Place 50:50 A/B + 5pp selector. Hidden/Restricted A probability with nature 0 guaranteed, nature 1–3 roll, weighted repair toward Hidden 2 / Restricted 3; nature 4 allowed only via strong-single-Faction fallback when target unmet. No upper cap. Access nodes do not modify geographic parent.

**Naming (2026-11-22):** 109 Animal-associated Flavor terms with weights 0/1/3. Single = Animal's Flavor Place; Dual = shared Flavor Place or original name; None = original name; City = Government Animal's Flavor City. Gate labels remain undecorated. Original IDs/base names preserved as separate from display_name.

**Impression tags (2026-11-22):** six multi-tags on 109 Flavor and 118 Place. For Single and City, tag overlap => weight 3, no overlap => 1; Item-only => 0. For Dual require shared Animal associations and tag-overlap 3; otherwise use base name. Original names/IDs and Gate nodes remain unchanged.

**Naming v2 final LOCK (2026-11-22):** Six impression multi-tags per Flavor/Place; only matching weight 3 can name a Dual-controlled Place. Final 22 Baby Animal words = Nature, Hiss = Mystery, Underworld = Mystery. 16 archived real generator seeds were locally replayed (568 Place decisions, including 18 named Dual), but upstream v0.4.3 selection + archived Sites is NOT a certified clean end-to-end hosted v0.4.8 runtime run. See `experiments/INFACTION_IMPRESSION_TAG_NAMING_QA_2026-11-22.md` for exact scope and examples.

**Seed behavior:** Every Run all creates ten distinct 63-bit stage seeds: Faction, Zone, Location, Function Repair, Placement, City, Site, Site Postprocessing, Access, and Naming. JSON includes `generation_seeds`; reproduce using all stage seeds. Running only export again reuses the current world.

**City Score hotfix:** City reads all 118 original Book1 City Scores including Town Hall. Export has run-identity checks to prevent mixed old/new Site data after a failed cell.


In [ ]:
import urllib.request,json,contextlib,io,types
ROOT="https://raw.githubusercontent.com/playrenathea/rantau/main/"
def get_text(path):
    # Cache-bust GitHub raw/CDN per fetch to avoid stale Python adapter hotfix.
    import uuid
    request=urllib.request.Request(ROOT+path+"?rev="+str(uuid.uuid4()),
        headers={"Cache-Control":"no-cache","Pragma":"no-cache"})
    return urllib.request.urlopen(request,timeout=60).read().decode("utf-8")
base=json.loads(get_text("notebooks/Infaction_Locked_City_Placement_v0.4.6.ipynb"))
# Discard previous Sites when starting a new world.
globals().pop("site_result",None)
globals().pop("_site_run_token",None)
globals().pop("full_world",None)
import uuid
_current_run_token=uuid.uuid4().hex
scope={"__name__":"__main__"}
for idx in (1,2,3):
    source_text="".join(base["cells"][idx]["source"])
    if idx==1:
        # The legacy notebook hardcoded Neutral 110; locked Neutral v2.1 has 118.
        legacy_assert="assert len(neutral)==len(city_scores)==110"
        assert legacy_assert in source_text, "Upstream preflight changed; recheck compatibility"
        source_text=source_text.replace(legacy_assert,
            "assert len(city_scores)==110 and len(neutral)==118 and set(city_scores).issubset(neutral)")
    if idx==3:
        # Independent random streams for each procedural stage.
        import secrets
        stages=("faction","zone","location","function_repair","placement","city","site","site_postprocess","access","naming")
        values=set()
        SEEDS={}
        for stage in stages:
            candidate=secrets.randbits(63)
            while candidate in values:
                candidate=secrets.randbits(63)
            values.add(candidate)
            SEEDS[stage]=candidate
        original='''SEED=8100000
world=ns['generate_world_with_locations'](SEED)
placed=placement.place_world(world,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=SEED)
result=city.city_place_world(placed,city_scores,seed=SEED)'''
        replacement=f'''SEEDS={SEEDS!r}
SEED=SEEDS["faction"]
integration_mod=types.ModuleType("infaction_v048_integrated_selection")
exec(compile(get_text("infaction_v048_integrated_selection.py"),
             "infaction_v048_integrated_selection.py","exec"),
     integration_mod.__dict__)
recovered=json.loads(get_text("data/infaction_missing_8_place_affinity_book1_recovered.json"))
location_setup=integration_mod.install_location_generation(
    ns,recovered,SITE_AFFINITY,SEEDS["function_repair"])
placement_setup=integration_mod.install_political_exception(
    placement,get_text("infaction_placement_locked.py"))
assert location_setup["eligible_location_count"]==71
world=ns["generate_world"](SEEDS["faction"])
world["zone_generation"]=ns["generate_zones"](world,seed=SEEDS["zone"])
world["location_generation"]=ns["generate_locations"](world,seed=SEEDS["location"])
assert world["location_generation"]["effective_pool"]==71
assert not world["location_generation"]["missing_minimum"]
assert len(world["location_generation"]["locations"])>=world["location_generation"]["target"]
world["version"]="0.4.8"
# The historical City notebook only loaded 110 scores; v0.4.8 needs all 118
# original Book1 city scores, including the new Town Hall Location.
city_scores={place:int(row["inside_city_original"])
             for place,row in MASTER["raw"]["place_catalog"].items()}
assert len(city_scores)==118, f"Expected 118 City Scores, got {len(city_scores)}"
placed=placement.place_world(world,ns["ZONE_DATA"],ns["LOCATION_DATA"],neutral,seed=SEEDS["placement"])
missing_city=(set(placed["zones"])|set(placed["locations"]))-set(city_scores)
assert not missing_city, f"Missing original City Score: {sorted(missing_city)}"
result=city.city_place_world(placed,city_scores,seed=SEEDS["city"])
print("Independent stage seeds:",SEEDS)'''
        assert source_text.count(original)==1, "Upstream generation cell changed; review seed-stage integration"
        source_text=source_text.replace(original,replacement)
    exec(compile(source_text,f"base_cell_{idx}","exec"),scope)
required=("world","placed","result","MASTER","SEED","SITE_AFFINITY","place_names")
missing=[x for x in required if x not in scope]
assert not missing, f"Legacy bootstrap missing: {missing}"
assert len(scope["world"]["factions"])==6
scope["run_token"]=_current_run_token
print("INTEGRATED v0.4.8: Neutral 118, Location 71, Function repair, political exception; seed:",scope["SEED"])


In [ ]:
from pathlib import Path
assert all(k in scope for k in ("world","placed","result","MASTER","SEED","run_token")), "Run bootstrap cell before Site cell"
_site_run_token=None
assert set(scope["placed"]["zones"])==set(scope["result"]["zones"]), "Placement/City Zone mismatch"
assert set(scope["placed"]["locations"])==set(scope["result"]["location_city"]), "Placement/City Location mismatch"
site_mod=types.ModuleType("infaction_site_proximity_checkpoint")
exec(compile(get_text("infaction_site_proximity_checkpoint.py"),"infaction_site_proximity_checkpoint.py","exec"),site_mod.__dict__)
master=scope["MASTER"]
scales={name:int(row["scale"]) for name,row in master["raw"]["place_catalog"].items()}
effective={**scales,**{"Fortress":3,"Palace":3,"Mine":3,"Factory":3,"Town Hall":2}}
site_candidates=[p for p in scope["place_names"] if effective.get(p) in (2,3,4)]
assert len(site_candidates)==50
site_city_scores={p:int(row["inside_city_original"]) for p,row in master["raw"]["place_catalog"].items()}
assert set(site_candidates).issubset(site_city_scores)
site_result=site_mod.group_sites(scope["placed"],scope["result"],scope["SITE_AFFINITY"],site_candidates,site_city_scores,seed=scope["SEEDS"]["site"])
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
family_mod=types.ModuleType("infaction_targeted_farm_ship_site_prune_v2")
exec(compile(get_text("experiments/infaction_targeted_farm_ship_site_prune_v2.py"),"infaction_targeted_farm_ship_site_prune_v2.py","exec"),family_mod.__dict__)
family_rules=json.loads(get_text("experiments/infaction_targeted_farm_ship_site_filter_v2.json"))
site_records=[{"site":s["name"],"container":("z:"+s["parent_zone"] if s["parent_zone"] is not None else s["city"].lower()),"children":s["children"]} for s in site_result["sites"]]
family_result=family_mod.prune_sites(site_records,family_rules,proximity=scope["SITE_AFFINITY"])
sites_by_name={s["name"]:s for s in site_result["sites"]}
retained_names={r["site"] for r in family_result["sites"]}
retained_sites=[{**sites_by_name[r["site"]],"children":r["children"]} for r in family_result["sites"]]
remaining_by_zone={z:list(members) for z,members in site_result["direct_locations_by_zone"].items()}
remaining_inside=list(site_result["standalone_ungrouped_inside"])
remaining_outside=list(site_result["standalone_ungrouped_outside"])
for container,children in family_result["direct_restored"].items():
    if container.startswith("z:"):
        remaining_by_zone[container[2:]].extend(children)
    elif container=="inside":
        remaining_inside.extend(children)
    elif container=="outside":
        remaining_outside.extend(children)
    else:
        raise AssertionError("Unknown restored container")
members=[loc for s in retained_sites for loc in s["children"]]
members+=sum(remaining_by_zone.values(),[])+remaining_inside+remaining_outside
assert len(members)==len(set(members))==len(scope["placed"]["locations"])
assert set(members)==set(scope["placed"]["locations"])
site_result["sites"]=retained_sites
site_result["direct_locations_by_zone"]=remaining_by_zone
site_result["standalone_ungrouped_inside"]=remaining_inside
site_result["standalone_ungrouped_outside"]=remaining_outside
site_result["grouped_location_count"]=sum(len(s["children"]) for s in retained_sites)
site_result["ungrouped_location_count"]=len(members)-site_result["grouped_location_count"]
site_result["family_filter"]={k:v for k,v in family_result.items() if k!="sites"}
site_result["status"]="v0.4.8 effective 71-location selection + Proximity + family postfilter v2"
# Post-Site rules operate after family prune, without rerunning the original greedy worlds.
post_mod=types.ModuleType("infaction_site_postprocess_v3")
exec(compile(get_text("infaction_site_postprocess_v3.py"),"infaction_site_postprocess_v3.py","exec"),post_mod.__dict__)
# The direct pools above already contain family-restored Locations: don't restore twice.
family_after={**family_result,"direct_restored":{}}
direct_pools={"z:"+z:list(ls) for z,ls in remaining_by_zone.items()}
direct_pools["inside"]=list(remaining_inside)
direct_pools["outside"]=list(remaining_outside)
post=post_mod.apply_garden_and_replacements(
    site_records,family_after,direct_pools,
    scope["placed"]["zones"],scope["placed"]["locations"],site_candidates,
    scope["SITE_AFFINITY"],site_city_scores,family_rules,scope["SEEDS"]["site_postprocess"])
def to_site_record(record):
    container=record["container"]
    zone=container[2:] if container.startswith("z:") else None
    city_label=scope["result"]["zones"][zone]["city"] if zone is not None else container.title()
    oldname="Garden" if record["site"]=="Grounds" else record["site"]
    original=sites_by_name.get(oldname,{})
    return {"name":record["site"],"parent_zone":zone,"city":city_label,
            "children":record["children"],"scores":original.get("scores",{
                "gap":record.get("gap"),"h2":record.get("h2"),"lowest":record.get("low")}),
            **({"replaces":record["replaces"]} if "replaces" in record else {})}
site_result["sites"]=[to_site_record(s) for s in post["sites"]]
unknown_site_zones=sorted({s["parent_zone"] for s in site_result["sites"]
                          if s["parent_zone"] is not None
                          and s["parent_zone"] not in scope["placed"]["zones"]})
assert not unknown_site_zones, f"Sites have foreign parent Zones: {unknown_site_zones}"
_site_run_token=scope["run_token"]
site_result["direct_locations_by_zone"]={z:post["direct_pools"]["z:"+z] for z in scope["placed"]["zones"]}
site_result["standalone_ungrouped_inside"]=post["direct_pools"]["inside"]
site_result["standalone_ungrouped_outside"]=post["direct_pools"]["outside"]
site_result["grouped_location_count"]=sum(len(s["children"]) for s in site_result["sites"])
site_result["ungrouped_location_count"]=len(scope["placed"]["locations"])-site_result["grouped_location_count"]
site_result["postprocessing"]={
    "version":post["version"],"garden_splits":post["garden_splits"],
    "replacement_sites":post["replacements"],"unreplaced":post["unreplaced"]}
site_result["status"]="Book1 Proximity + Farm/Ship family v2 + Garden/Grounds split + optional replacement"
assert site_result["grouped_location_count"]+site_result["ungrouped_location_count"]==len(scope["placed"]["locations"])
print("Seed",scope["SEED"],"Sites",len(site_result["sites"]),
      "Garden split",len(post["garden_splits"]),
      "replacements",len(post["replacements"]),"unreplaced",len(post["unreplaced"]))
for s in site_result["sites"]:
    print(s["name"],"in",s["parent_zone"] or "Standalone",":",", ".join(s["children"]))
Path(f'infaction_world_with_sites_experimental_{scope["SEED"]}.json').write_text(
    json.dumps({"world":scope["world"],"placement":scope["placed"],
                "city":scope["result"],"site_grouping":site_result},
               indent=2,ensure_ascii=False))


In [ ]:
# Export a complete, navigable world Place tree and full Faction attributes.
# Auto-bootstrap on a fresh runtime / when this cell is run directly.
if ('scope' not in globals() or not isinstance(scope,dict) or
        any(k not in scope for k in ('world','placed','result','MASTER','SEED','run_token')) or
        'site_result' not in globals() or
        '_site_run_token' not in globals() or _site_run_token!=scope['run_token']):
    _current=json.loads(urllib.request.urlopen('https://raw.githubusercontent.com/playrenathea/rantau/main/notebooks/Infaction_v0.4.8_World_Access_HOTFIX_20261122.ipynb').read().decode('utf-8')) if 'urllib' in globals() else None
    if _current is None:
        import urllib.request,json,types
        _current=json.loads(urllib.request.urlopen('https://raw.githubusercontent.com/playrenathea/rantau/main/notebooks/Infaction_v0.4.8_World_Access_HOTFIX_20261122.ipynb').read().decode('utf-8'))
    for _idx in (1,2):
        exec(compile(''.join(_current['cells'][_idx]['source']),f'bootstrap_cell_{_idx}','exec'))
assert 'world' in scope and 'site_result' in globals(), 'Bootstrap incomplete: run all cells'
assert _site_run_token==scope['run_token'], 'Stale Site data: rerun notebook from cell 1'
assert set(scope['placed']['zones'])==set(scope['result']['zones'])
assert all(s['parent_zone'] is None or s['parent_zone'] in scope['placed']['zones']
           for s in site_result['sites']), 'Site Zone outside current generated world'
export_mod=types.ModuleType('infaction_world_json_export_v048')
exec(compile(get_text('infaction_world_json_export_v048.py'),'infaction_world_json_export_v048.py','exec'),export_mod.__dict__)
full_world=export_mod.build_world_export(
    scope['world'], scope['placed'], scope['result'], site_result,
    place_catalog=scope['MASTER']['raw']['place_catalog'])
assert len(full_world['factions'])==6
assert len(full_world['places']['nodes']) == 2+len(scope['placed']['zones'])+len(scope['placed']['locations'])+len(site_result['sites'])
assert all('base_affinities' in f and 'animal_affinities' in f and 'types' in f for f in full_world['factions'])
# Add preserve-first political ownership and lightweight Hidden/Restricted access.
neutral_mod_data=json.loads(get_text("data/infaction_neutral_static_v2_locked.json"))
neutral118={r["place"]:r["neutral_affinity"] for r in neutral_mod_data["records"]}
assert len(neutral118)==118 and neutral_mod_data["version"]=="neutral-static-v2.1"
access_mod=types.ModuleType("infaction_hidden_restricted_locked")
exec(compile(get_text("infaction_hidden_restricted_locked.py"),
             "infaction_hidden_restricted_locked.py","exec"),access_mod.__dict__)
world_mod=types.ModuleType("infaction_ownership_hidden_restricted_locked")
exec(compile(get_text("infaction_ownership_hidden_restricted_locked.py"),
             "infaction_ownership_hidden_restricted_locked.py","exec"),world_mod.__dict__)
# Runtime-specific compatibility bridge for the generated Garden -> Grounds Site.
# Grounds is an output alias, not a selectable 119th Book1 Place type.
_catalog = scope["MASTER"]["raw"]["place_catalog"]
if any(node["generation_role"]=="site" and node["name"]=="Grounds"
       for node in full_world["places"]["nodes"].values()):
    assert "Garden" in _catalog, "Garden nature missing from original Book1"
    if "Grounds" not in _catalog:
        _catalog["Grounds"] = dict(_catalog["Garden"])
    assert _catalog["Grounds"]["visible"] == _catalog["Garden"]["visible"]
    assert _catalog["Grounds"]["accessible_outsider"] == _catalog["Garden"]["accessible_outsider"]
    print("GROUNDS ALIAS ACTIVE: inherited Garden nature for generated Grounds")
refs=world_mod.source_reference(scope["MASTER"])
owners=world_mod.preserve_ownership(full_world,refs,neutral118)
# Access uses its own independent seed; restore the canonical faction seed afterwards.
_canonical_seed=full_world["seed"]
try:
    full_world["seed"]=scope["SEEDS"]["access"]
    full_world=world_mod.attach_access(full_world,owners,neutral118,scope["MASTER"],access_mod)
finally:
    full_world["seed"]=_canonical_seed
assert len(full_world["political_ownership"]["places"]) == (
    len(scope["placed"]["zones"])+len(scope["placed"]["locations"])+len(site_result["sites"]))
assert all("political_ownership" in node["attributes"]
           for node in full_world["places"]["nodes"].values()
           if node["generation_role"] in ("zone","site","location"))
print("Ownership decision counts:",owners["counts"])
print("Access status:",full_world["hidden_restricted"]["summary"])
# Final naming overlay: leave original Place identities and Gate labels unchanged.
name_data=json.loads(get_text("data/infaction_animal_place_flavor_v1_locked.json"))
name_mod=types.ModuleType("infaction_place_naming_locked")
exec(compile(get_text("infaction_place_naming_locked.py"),
             "infaction_place_naming_locked.py","exec"),name_mod.__dict__)
impression_data=json.loads(get_text("data/infaction_place_impression_tags_v1.json"))
# Naming uses a different independent seed without changing world identity.
try:
    full_world["seed"]=scope["SEEDS"]["naming"]
    full_world=name_mod.apply(full_world,name_data,impression_data)
finally:
    full_world["seed"]=_canonical_seed
full_world["generation_seeds"]=dict(scope["SEEDS"])
full_world["status"]["upstream_runtime"]="v0.4.8 effective integrated selection: 71 Location types, Function minimum repair, political post-choice exception; live hosted-Colab clean-run not yet observed"
full_world["status"]["runtime_integration"]="infaction_v048_integrated_selection.py; source Book1 unchanged"
full_world["status"]["random_streams"]="10 independent stage seeds in generation_seeds"
assert full_world["generation_seeds"]["function_repair"] != full_world["generation_seeds"]["location"]
assert not scope["world"]["location_generation"]["missing_minimum"]
assert len(scope["world"]["location_generation"]["locations"])==len(scope["placed"]["locations"])
assert len(set(scope["placed"]["locations"]))==len(scope["placed"]["locations"])
assert all(name not in {"Fortress","Palace","Mine","Factory"} for name in scope["placed"]["locations"])
assert full_world["place_naming"]["summary"]["city"] == 1
assert full_world["place_naming"]["impression_version"] == "impression-tags-v1-2026-11-22"
assert all(p["name"]==p["base_name"] and p["display_name"]
           for p in full_world["places"]["nodes"].values())
print("Place naming summary:",full_world["place_naming"]["summary"])
export_path=Path(f"infaction_world_named_seed_{scope['SEED']}_v0.4.8-access.json")
export_mod.save_world_json(full_world,export_path)
print('PASS JSON:',export_path,'size',export_path.stat().st_size,'bytes')
print('Faction count:',len(full_world['factions']))
print('Place counts:',full_world['places']['counts'])
print('Root children:',[full_world['places']['nodes'][i]['name'] for i in full_world['places']['nodes']['land']['children_ids']])
print('Runtime status:',full_world['status']['upstream_runtime'])
# Download the saved JSON from Colab; this does not affect model or sources.
try:
    from google.colab import files
    files.download(str(export_path))
except ImportError:
    print('Saved locally:',export_path.resolve())
